# Vector DB Analysis
Quick inspection of the Chroma vector database built by `vectorDB.py`.

In [3]:
import chromadb
from pathlib import Path

DB_PATH = Path("db/chroma")
client = chromadb.PersistentClient(path=str(DB_PATH))

# List all collections
collections = client.list_collections()
print(f"Collections: {[c.name for c in collections]}")

Collections: ['documents']


In [4]:
# Pick the collection
col = client.get_collection(collections[0].name)

# 1. Total rows
print(f"Total records: {col.count()}")

# 2. Embedding dimensions
sample = col.peek(limit=1)
if sample["embeddings"]:
    print(f"Embedding dimensions: {len(sample['embeddings'][0])}")
else:
    sample_with_emb = col.get(limit=1, include=["embeddings"])
    print(f"Embedding dimensions: {len(sample_with_emb['embeddings'][0])}")

Total records: 5624


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [3]:
# 3. Content type breakdown (text vs image vs table)
all_meta = col.get(include=["metadatas"])["metadatas"]

from collections import Counter
type_counts = Counter(m.get("content_type", "unknown") for m in all_meta)
clip_count = sum(1 for m in all_meta if m.get("clip_image_embedded"))

print("Content type breakdown:")
for ctype, count in type_counts.most_common():
    print(f"  {ctype}: {count}")
if clip_count:
    print(f"  (of which {clip_count} are CLIP image-embedded)")

Content type breakdown:


In [4]:
# 4. Metadata structure + sample record
all_keys = set()
for m in all_meta:
    all_keys.update(m.keys())
print(f"Metadata fields: {sorted(all_keys)}")

print("\n--- Sample record ---")
sample_full = col.peek(limit=1)
print(f"ID:       {sample_full['ids'][0]}")
print(f"Metadata: {sample_full['metadatas'][0]}")
print(f"Text:     {sample_full['documents'][0][:300]}...")

Metadata fields: []

--- Sample record ---


IndexError: list index out of range

In [ ]:
# 5. Source file breakdown
source_counts = Counter(Path(m.get("source", "unknown")).name for m in all_meta)

print(f"Documents indexed: {len(source_counts)}")
print("\nChunks per source file:")
for src, count in source_counts.most_common():
    print(f"  {src}: {count}")